# Agentic AI — Maintenance Assistant

**Goal:** Build an LLM agent that routes user questions to the right tool:
- predict_failure: for questions about specific machine sensor readings
- search_docs: for questions about maintenance procedures and guidelines

**Pattern:** ReAct (Reason + Act) — agent reasons before each action
and continues until it has enough information to answer.

**Why this matters:** Unlike fixed pipelines, the agent adapts its approach based on the question — calling one tool, multiple tools, or no tool depending on what the user actually needs.

In [1]:
import requests
import json
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path to import RAG components
sys.path.append(str(Path('..').resolve()))

load_dotenv('../2_rag/.env')

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
API_BASE_URL = "http://127.0.0.1:8000"

print("Setup complete")
print(f"API URL: {API_BASE_URL}")
print(f"Gemini key loaded: {'yes' if GOOGLE_API_KEY else 'NO - check .env file'}")

Setup complete
API URL: http://127.0.0.1:8000
Gemini key loaded: yes


## Build the tools

### Tool 1 — predict_failure:

In [2]:
def predict_failure(
    machine_type: str,
    air_temperature: float,
    process_temperature: float,
    rotational_speed: float,
    torque: float,
    tool_wear: float
) -> dict:
    """
    Predicts machine failure probability from sensor readings.
    Call this when the user provides specific sensor values for a machine.

    Returns failure probability, prediction label, and risk level.
    """
    payload = {
        "type": machine_type.upper(),
        "air_temperature": air_temperature,
        "process_temperature": process_temperature,
        "rotational_speed": rotational_speed,
        "torque": torque,
        "tool_wear": tool_wear
    }

    try:
        response = requests.post(
            f"{API_BASE_URL}/predict",
            json=payload,
            timeout=10
        )
        response.raise_for_status()
        result = response.json()
        return {
            "status": "success",
            "failure_probability": result["failure_probability"],
            "prediction": result["prediction"],
            "risk_level": result["risk_level"],
            "interpretation": f"Machine has {result['failure_probability']*100:.1f}% failure probability — {result['risk_level']} risk"
        }
    except requests.exceptions.ConnectionError:
        return {"status": "error", "message": "API not reachable. Make sure FastAPI server is running on port 8000."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Test it
result = predict_failure("L", 298.1, 308.6, 1363.0, 68.0, 220.0)
print(json.dumps(result, indent=2))

{
  "status": "success",
  "failure_probability": 1.0,
  "prediction": "FAILURE",
  "risk_level": "HIGH",
  "interpretation": "Machine has 100.0% failure probability \u2014 HIGH risk"
}


### Tool 2 — search_docs:

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load the FAISS index built in Session 4
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "../2_rag/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

def search_docs(query: str, k: int = 3) -> dict:
    """
    Searches maintenance documentation for relevant information.
    Call this when the user asks general questions about maintenance
    procedures, failure causes, equipment guidelines, or best practices.

    Returns relevant passages from the maintenance manual.
    """
    try:
        docs = vectorstore.similarity_search(query, k=k)

        if not docs:
            return {
                "status": "success",
                "results": [],
                "message": "No relevant documents found for this query."
            }

        results = []
        for i, doc in enumerate(docs):
            results.append({
                "rank": i + 1,
                "content": doc.page_content,
                "source": doc.metadata.get("source", "maintenance_manual")
            })

        return {
            "status": "success",
            "query": query,
            "results": results,
            "summary": f"Found {len(results)} relevant passages"
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

# Test it
result = search_docs("Why are particles smaller than 5 microns considered more dangerous for hydraulic systems than larger particles?")
print(json.dumps(result, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "status": "success",
  "query": "Why are particles smaller than 5 microns considered more dangerous for hydraulic systems than larger particles?",
  "results": [
    {
      "rank": 1,
      "content": "Particles smaller than 5 microns are highly abrasive. If present in sufficient quantities, these \ninvisible 'silt' particles cause rapid wear, destroying hydraulic components. \n \n \nQuantifying Particle Contamination \n \nSome level of particle contamination is always present in hydraulic fluid, even in new fluid. It \nis the size and quantity of these particles that we are concerned with. The level of",
      "source": "docs/maintenance_guide.pdf"
    },
    {
      "rank": 2,
      "content": "CLEARANCE IN MICRONS \nGear pump 0.5 \u2013 5.0 \nVane pump 0.5 \u2013 10 \nPiston pump 0.5 \u2013 5.0 \nServo valve 1.0 \u2013 4.0 \nControl valve 0.5 \u2013 40 \nLinear actuator 50 - 250 \n \nParticles larger than a component's internal clearances are not necessarily dangerous. Particle

In [5]:
# Health check — agent will fail silently if tools are broken
api_health = requests.get(f"{API_BASE_URL}/health").json()
print(f"API health: {api_health}")

test_prediction = predict_failure("M", 298.1, 308.6, 1551.0, 42.8, 0.0)
print(f"Prediction tool: {test_prediction['status']}")

test_search = search_docs("tool wear replacement")
print(f"Search tool: {test_search['status']}")

print("\nAll tools operational ✅" if all(
    t['status'] == 'success'
    for t in [test_prediction, test_search]
) else "\n⚠️ Fix tool errors before proceeding")

API health: {'status': 'healthy', 'model_loaded': True}
Prediction tool: success
Search tool: success

All tools operational ✅


### Build the ReAct agent

In [12]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

tools = [
    genai.protos.Tool(
        function_declarations=[
            genai.protos.FunctionDeclaration(
                name="predict_failure",
                description="""Predicts machine failure probability from sensor readings.
                Use this tool when the user provides specific sensor values such as
                torque, tool wear, rotational speed, or temperature readings for a machine.
                Do NOT use for general maintenance questions.""",
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        "machine_type": genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description="Machine quality tier: L (low), M (medium), or H (high)"
                        ),
                        "air_temperature": genai.protos.Schema(
                            type=genai.protos.Type.NUMBER,
                            description="Ambient air temperature in Kelvin"
                        ),
                        "process_temperature": genai.protos.Schema(
                            type=genai.protos.Type.NUMBER,
                            description="Process temperature in Kelvin"
                        ),
                        "rotational_speed": genai.protos.Schema(
                            type=genai.protos.Type.NUMBER,
                            description="Rotational speed in RPM"
                        ),
                        "torque": genai.protos.Schema(
                            type=genai.protos.Type.NUMBER,
                            description="Torque applied in Newton-meters"
                        ),
                        "tool_wear": genai.protos.Schema(
                            type=genai.protos.Type.NUMBER,
                            description="Tool wear duration in minutes"
                        ),
                    },
                    required=["machine_type", "air_temperature", "process_temperature",
                              "rotational_speed", "torque", "tool_wear"]
                )
            ),
            genai.protos.FunctionDeclaration(
                name="search_docs",
                description="""Searches maintenance documentation for relevant information.
                Use this tool for general questions about maintenance procedures,
                failure causes, equipment guidelines, safety protocols, or best practices.
                Do NOT use when specific sensor readings are provided.""",
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        "query": genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description="The search query to find relevant maintenance information"
                        ),
                    },
                    required=["query"]
                )
            )
        ]
    )
]

#### Agent Loop

In [15]:
def run_agent(user_question: str, max_iterations: int = 5) -> str:
    """
    ReAct agent that routes questions to the appropriate tool.

    Args:
        user_question: The user's question
        max_iterations: Safety limit to prevent infinite loops

    Returns:
        Final answer string
    """
    print(f"\n{'='*60}")
    print(f"USER: {user_question}")
    print(f"{'='*60}")

    # Initialize Gemini with function calling
    model = genai.GenerativeModel(
        model_name="gemini-2.0-flash",
        tools=tools,
        system_instruction="""You are an expert maintenance engineer assistant.
        You help diagnose machine failures and answer maintenance questions.

        You have access to two tools:
        1. predict_failure: for analyzing specific machine sensor readings
        2. search_docs: for answering general maintenance questions

        Always reason about which tool is appropriate before calling it.
        If sensor values are provided, use predict_failure.
        If the question is general, use search_docs.
        For questions requiring both, call both tools."""
    )

    chat = model.start_chat()
    messages = [user_question]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")

        # Send message to LLM
        response = chat.send_message(messages[-1] if iteration == 1 else "Continue based on the tool result.")

        # Check if LLM wants to call a tool
        if response.candidates[0].content.parts[0].function_call.name if hasattr(response.candidates[0].content.parts[0], 'function_call') and response.candidates[0].content.parts[0].function_call.name else None:

            for part in response.candidates[0].content.parts:
                if hasattr(part, 'function_call') and part.function_call.name:
                    tool_name = part.function_call.name
                    tool_args = dict(part.function_call.args)

                    print(f"THOUGHT: Agent decided to call → {tool_name}")
                    print(f"ARGS: {json.dumps(tool_args, indent=2)}")

                    # Execute the tool
                    if tool_name == "predict_failure":
                        tool_result = predict_failure(**tool_args)
                    elif tool_name == "search_docs":
                        tool_result = search_docs(**tool_args)
                    else:
                        tool_result = {"error": f"Unknown tool: {tool_name}"}

                    print(f"OBSERVATION: {json.dumps(tool_result, indent=2)[:300]}...")

                    # Send result back to LLM
                    messages.append(f"Tool result from {tool_name}: {json.dumps(tool_result)}")

        else:
            # LLM gave a final text answer — no more tool calls
            final_answer = response.text
            print(f"\nFINAL ANSWER: {final_answer}")
            return final_answer

    return "Max iterations reached without a final answer."

In [18]:
import time

def run_agent_with_retry(question, retries=3, wait=15):
    for attempt in range(retries):
        try:
            return run_agent(question)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                if attempt < retries - 1:
                    print(f"Rate limit hit. Waiting {wait}s before retry {attempt+1}/{retries}...")
                    time.sleep(wait)
                else:
                    print("❌ Rate limit reached after 3 attempts. Stopping execution.")
                    raise SystemExit("Stopping: Gemini API rate limit exceeded. Wait a few minutes and try again.")
            else:
                raise e

test_cases = [
    "My L-type machine has air temp 298K, process temp 308K, rotational speed 1363 rpm, torque 68 Nm and tool wear 220 min. Should I be worried?",
    "If a fluid sample from a normal-pressure system shows an ISO cleanliness level of 19/16, what steps should be taken to rectify the contamination?",
    "According to Exhibit 1.1, what are the typical internal clearances (in microns) for piston pumps, servo valves, and linear actuators?",
    "Assess this machine: type M, air temp 300K, process temp 310K, speed 1500 rpm, torque 45 Nm, wear 90 min",
    "My machine has torque 68 Nm and tool wear 220 min. Is it going to fail, and For a high-pressure hydraulic system (250–400 bar), what is the minimum recommended cleanliness level in ISO 4406, and what filtration level (in microns) is required to achieve it??",
    "What is the weather like in Paris today?"
]

results = []
for i, question in enumerate(test_cases):
    print(f"Running test case {i+1}/{len(test_cases)}...")
    answer = run_agent_with_retry(question)
    results.append({"question": question, "answer": answer})
    print("\n")
    if i < len(test_cases) - 1:
        print("Waiting 10s before next question...")
        time.sleep(10)

Running test case 1/6...

USER: My L-type machine has air temp 298K, process temp 308K, rotational speed 1363 rpm, torque 68 Nm and tool wear 220 min. Should I be worried?

--- Iteration 1 ---
Rate limit hit. Waiting 15s before retry 1/3...

USER: My L-type machine has air temp 298K, process temp 308K, rotational speed 1363 rpm, torque 68 Nm and tool wear 220 min. Should I be worried?

--- Iteration 1 ---
Rate limit hit. Waiting 15s before retry 2/3...

USER: My L-type machine has air temp 298K, process temp 308K, rotational speed 1363 rpm, torque 68 Nm and tool wear 220 min. Should I be worried?

--- Iteration 1 ---
❌ Rate limit reached after 3 attempts. Stopping execution.


SystemExit: Stopping: Gemini API rate limit exceeded. Wait a few minutes and try again.

/opt/anaconda3/envs/data_env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
